## 处理手工数据，形成csv文件

In [ ]:
import sys
import pandas as pd


def excel_to_csv(excel_path, csv_path):
    try:
        # 读取 Excel
        df = pd.read_excel(excel_path, engine="openpyxl")

        # 需要保留的列
        columns = [
            "source_title",
            "source_content",
            "is_show"
        ]

        # 检查列是否存在
        missing_columns = [col for col in columns if col not in df.columns]
        if missing_columns:
            print(f"缺少以下列：{missing_columns}")
            return

        # 只保留指定列
        df = df[columns]

        # 修改列名
        df = df.rename(columns={
            "source_title": "title",
            "source_content": "content",
            "is_show": "label"
        })

        # 保存 CSV
        # utf-8-sig 可以避免中文在 Excel 中打开乱码
        df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"转换完成：{csv_path}")
        print(f"共导出 {len(df)} 条数据")

        print("\n前 5 条数据：")
        print(df.head())

    except FileNotFoundError:
        print(f"找不到文件：{excel_path}")

    except Exception as e:
        print(f"处理失败：{e}")



excel_file = "/mnt/cognitive_classify_model/dqdx_hand.xlsx"
csv_file = "/mnt/cognitive_classify_model/dqdx_hand.xlsx"

excel_to_csv(excel_file, csv_file)

In [3]:
import pandas as pd


def build_dataset(
    file1_path="file1.xlsx",
    file2_path="file2.csv",
    output_path="result.csv"
):
    # =========================
    # 1. 处理 file1.xlsx
    # =========================
    df1 = pd.read_excel(file1_path, engine="openpyxl")

    # 检查需要的字段
    required_columns1 = ["id", "source_title", "source_content"]

    missing_columns1 = [
        col for col in required_columns1
        if col not in df1.columns
    ]

    if missing_columns1:
        raise ValueError(
            f"{file1_path} 缺少字段：{missing_columns1}"
        )

    # source_content 不为空
    # 同时排除 NaN、空字符串、只有空格的情况
    df1 = df1[
        df1["source_content"].notna()
        & (df1["source_content"].astype(str).str.strip() != "")
    ].copy()

    # 保留指定字段
    df1 = df1[
        ["id", "source_title", "source_content"]
    ]

    # 修改字段名
    df1 = df1.rename(columns={
        "source_title": "title",
        "source_content": "content"
    })

    # 添加 label
    df1["label"] = "有用"

    useful_count = len(df1)

    print(f"file1 有用数据数量：{useful_count}")

    if useful_count == 0:
        raise ValueError("file1.xlsx 中没有 source_content 非空的数据")

    # =========================
    # 2. 处理 file2.csv
    # =========================
    df2 = pd.read_csv(file2_path)

    required_columns2 = [
        "id",
        "title",
        "content",
        "is_show"
    ]

    missing_columns2 = [
        col for col in required_columns2
        if col not in df2.columns
    ]

    if missing_columns2:
        raise ValueError(
            f"{file2_path} 缺少字段：{missing_columns2}"
        )

    # 筛选 is_show 为空
    # 同时处理 NaN、空字符串、只有空格的情况
    df2 = df2[
        df2["is_show"].isna()
        | (df2["is_show"].astype(str).str.strip() == "")
    ].copy()

    useless_available_count = len(df2)

    print(f"file2 中 is_show 为空的数据数量：{useless_available_count}")

    # 判断数据量是否足够
    if useless_available_count < useful_count:
        raise ValueError(
            f"file2 中 is_show 为空的数据不足。\n"
            f"需要：{useful_count} 条\n"
            f"实际只有：{useless_available_count} 条"
        )

    # 随机抽取和 file1 相同数量的数据
    # random_state 保证每次运行抽到的数据一致
    df2 = df2.sample(
        n=useful_count,
        random_state=42
    )

    # 只保留指定字段
    df2 = df2[
        ["id", "title", "content"]
    ]

    # 添加 label
    df2["label"] = "无用"

    print(f"file2 抽取无用数据数量：{len(df2)}")

    # =========================
    # 3. 合并
    # =========================
    result = pd.concat(
        [df1, df2],
        ignore_index=True
    )

    # 可选：打乱顺序
    result = result.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    # =========================
    # 4. 保存 CSV
    # =========================
    result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n处理完成")
    print(f"有用数据：{len(df1)}")
    print(f"无用数据：{len(df2)}")
    print(f"总数据量：{len(result)}")
    print(f"输出文件：{output_path}")

    print("\n前 10 条数据：")
    print(result.head(10))


build_dataset(
    file1_path="/mnt/cognitive_classify_model/data_process/dqdx_hand.xlsx",
    file2_path="/mnt/cognitive_classify_model/data_process/hits_clean.csv",
    output_path="/mnt/cognitive_classify_model/data_process/train_data.csv"
)

file1 有用数据数量：212
file2 中 is_show 为空的数据数量：941
file2 抽取无用数据数量：212

处理完成
有用数据：212
无用数据：212
总数据量：424
输出文件：/mnt/cognitive_classify_model/data_process/train_data.csv

前 10 条数据：
                                 id  \
0               2007119525135597568   
1  859cdd70b35e1efe891869a5218bdcb3   
2               1995506997031170048   
3  26f4187f90b463f1709b380d61c425ab   
4  b415c74f33e3cd2404f3eeca6d61cc24   
5               2043338548189040640   
6               2013980125128114176   
7               2012154589531684864   
8               2067262419496558592   
9               2043688504108224512   

                                               title  \
0                 The AFP @ 90 and cognitive defense   
1  The App Store is booming again, and AI may be why   
2  Rebalancing the Information Ecosystem and Rene...   
3  Portugal warns foreign state-backed hackers ta...   
4  Israeli forces intercept Global Sumud flotilla...   
5                               「台灣熱新聞」創刊 盼加深日本對台灣理解   
6  Cogn

In [7]:
import json
import random
import pandas as pd


def is_empty(value):
    """
    判断字段是否为空。
    None、""、"   " 都认为是空。
    """
    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    return False


def build_dataset(
    file1_path="file1.xlsx",
    file2_path="DQDX_no_hand.json",
    output_path="result.jsonl"
):
    # ==================================================
    # 1. 读取 file1.xlsx
    # ==================================================
    df1 = pd.read_excel(
        file1_path,
        engine="openpyxl",
        dtype=str,
        keep_default_na=False
    )

    required_columns = [
        "id",
        "source_title",
        "source_content"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df1.columns
    ]

    if missing_columns:
        raise ValueError(
            f"file1.xlsx 缺少字段：{missing_columns}"
        )

    # ==================================================
    # 2. source_content 不为空
    # ==================================================
    df1 = df1[
        df1["source_content"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    # 只保留三个字段
    df1 = df1[
        [
            "id",
            "source_title",
            "source_content"
        ]
    ].copy()

    # 修改名称
    df1.rename(
        columns={
            "source_title": "title",
            "source_content": "content"
        },
        inplace=True
    )

    # label = 有用
    df1["label"] = "有用"

    useful_count = len(df1)

    print(
        f"file1 中 source_content 非空："
        f"{useful_count} 条"
    )

    if useful_count == 0:
        raise ValueError(
            "file1.xlsx 没有符合条件的数据"
        )

    # ==================================================
    # 3. 读取原始 JSON
    # ==================================================
    with open(
        file2_path,
        "r",
        encoding="utf-8"
    ) as f:
        json_data = json.load(f)

    # Elasticsearch 结构：
    # json_data["hits"]["hits"]
    hits = (
        json_data
        .get("hits", {})
        .get("hits", [])
    )

    print(
        f"JSON 中实际读取到：{len(hits)} 条记录"
    )

    # ==================================================
    # 4. 筛选 is_show 缺省或者为空
    # ==================================================
    useless_data = []

    for hit in hits:

        source = hit.get("_source", {})

        # --------------------------------------------
        # 情况1：
        # 根本不存在 is_show
        # --------------------------------------------
        if "is_show" not in source:
            valid = True

        # --------------------------------------------
        # 情况2：
        # is_show 存在，但为 None / "" / 空格
        # --------------------------------------------
        else:
            valid = is_empty(
                source.get("is_show")
            )

        if not valid:
            continue

        # 读取需要的字段
        item = {
            "id": str(
                source.get("id", "")
            ),
            "title": str(
                source.get("title", "")
            ),
            "content": str(
                source.get("content", "")
            ),
            "label": "无用"
        }

        useless_data.append(item)

    print(
        f"is_show 缺省或为空："
        f"{len(useless_data)} 条"
    )

    # ==================================================
    # 5. 检查数量
    # ==================================================
    if len(useless_data) < useful_count:
        raise ValueError(
            "\nJSON 中符合条件的数据不够：\n"
            f"file1 需要：{useful_count} 条\n"
            f"JSON 可用：{len(useless_data)} 条"
        )

    # ==================================================
    # 6. 随机抽取与 file1 一样多
    # ==================================================
    random.seed(42)

    useless_data = random.sample(
        useless_data,
        useful_count
    )

    print(
        f"最终抽取无用数据："
        f"{len(useless_data)} 条"
    )

    # ==================================================
    # 7. file1 转成 list
    # ==================================================
    useful_data = df1.to_dict(
        orient="records"
    )

    # 保证都是普通字符串
    for item in useful_data:

        item["id"] = str(
            item.get("id", "")
        )

        item["title"] = str(
            item.get("title", "")
        )

        item["content"] = str(
            item.get("content", "")
        )

        item["label"] = "有用"

    # ==================================================
    # 8. 合并
    # ==================================================
    result = (
        useful_data
        +
        useless_data
    )

    print(
        f"合并后总数据量："
        f"{len(result)} 条"
    )

    # ==================================================
    # 9. 随机打乱
    # ==================================================
    random.seed(42)

    random.shuffle(result)

    # ==================================================
    # 10. 保存 JSONL
    # ==================================================
    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for item in result:

            line = json.dumps(
                item,

                # 中文不要变成 \uXXXX
                ensure_ascii=False
            )

            f.write(
                line + "\n"
            )

    print("\n==============================")
    print("处理完成")
    print("==============================")

    print(
        f"有用数据：{len(useful_data)}"
    )

    print(
        f"无用数据：{len(useless_data)}"
    )

    print(
        f"总数据：{len(result)}"
    )

    print(
        f"输出文件：{output_path}"
    )




build_dataset(
    file1_path="/mnt/cognitive_classify_model/data_process/dqdx_hand.xlsx",

    # 现在直接使用原始 JSON
    file2_path="/mnt/cognitive_classify_model/data_process/DQDX_no_hand.json",

    output_path="./result.jsonl"
)

file1 中 source_content 非空：212 条
JSON 中实际读取到：1000 条记录
is_show 缺省或为空：941 条
最终抽取无用数据：212 条
合并后总数据量：424 条

处理完成
有用数据：212
无用数据：212
总数据：424
输出文件：./result.jsonl


## 读取DQDX_no_hand.json中的数据，"is_show"为2的数据，读取字段为"id","title","content",同样添加一个字段"label",将这里的数据的"label"设置为有用,然后接着读取train.jsonl中的数据，读取字段为"id",在DQDX_no_hand.json中的数据取出不包含id在train.jsonl的id中且"is_show"为空的数据，数据个数和"is_show"为2的数量一致，读取字段为"id","title","content","label"设置为无用。形成一个新的jsonl

In [1]:
import json
import random


def normalize_id(value):
    if value is None:
        return ""
    return str(value).strip()


def is_show_equal_2(source):
    """
    兼容:
    "is_show": 2
    "is_show": "2"
    """
    if "is_show" not in source:
        return False

    value = source.get("is_show")

    if value is None:
        return False

    return str(value).strip() == "2"


def is_show_empty(source):
    """
    以下情况都认为 is_show 为空：
    - 字段不存在
    - null
    - ""
    - "   "
    """
    if "is_show" not in source:
        return True

    value = source.get("is_show")

    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    return False


def clean_text(value):
    if value is None:
        return ""

    value = str(value)

    # 去掉 NULL 字符
    value = value.replace("\x00", "")

    # 统一换行
    value = value.replace("\r\n", "\n")
    value = value.replace("\r", "\n")

    return value


def read_jsonl_ids(file_path):
    """
    读取 file3.jsonl 中所有 id
    """
    ids = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                print(
                    f"警告：file3 第 {line_number} 行 JSON 解析失败，已跳过：{e}"
                )
                continue

            current_id = normalize_id(
                item.get("id")
            )

            if current_id:
                ids.add(current_id)

    return ids


def build_dataset(
    file2_path="file2.json",
    file3_path="file3.jsonl",
    output_path="result.jsonl",
    random_seed=42
):
    """
    逻辑：

    1. file2.json 中 is_show = 2
       -> label = 有用

    2. 读取 file3.jsonl 的所有 id

    3. file2.json 中筛选：
       - is_show 为空或不存在
       - id 不在 file3.jsonl 中

    4. 从无用候选里随机抽取，
       数量和 is_show=2 的数量一致

    5. label = 无用

    6. 合并并打乱

    7. 保存为 JSONL
    """

    # ==========================================
    # 1. 读取 file3.jsonl 中的 id
    # ==========================================

    print("正在读取 file3.jsonl ...")

    file3_ids = read_jsonl_ids(
        file3_path
    )

    print(
        f"file3.jsonl 中读取到 {len(file3_ids)} 个不同 id"
    )

    # ==========================================
    # 2. 读取 file2.json
    # ==========================================

    print("\n正在读取 file2.json ...")

    with open(
        file2_path,
        "r",
        encoding="utf-8"
    ) as f:
        json_data = json.load(f)

    hits = (
        json_data
        .get("hits", {})
        .get("hits", [])
    )

    print(
        f"file2.json 总数据量：{len(hits)} 条"
    )

    # ==========================================
    # 3. 提取 is_show = 2
    # ==========================================

    useful_data = []

    for hit in hits:

        source = hit.get("_source", {})

        if not is_show_equal_2(source):
            continue

        current_id = normalize_id(
            source.get("id")
        )

        item = {
            "id": current_id,
            "title": clean_text(
                source.get("title")
            ),
            "content": clean_text(
                source.get("content")
            ),
            "label": "有用"
        }

        useful_data.append(item)

    useful_count = len(useful_data)

    print(
        f"is_show = 2 的有用数据：{useful_count} 条"
    )

    if useful_count == 0:
        raise ValueError(
            "file2.json 中没有找到 is_show = 2 的数据"
        )

    # ==========================================
    # 4. 找无用候选
    #
    # 条件：
    # is_show 为空
    # 且 id 不在 file3.jsonl
    # ==========================================

    useless_candidates = []

    for hit in hits:

        source = hit.get("_source", {})

        if not is_show_empty(source):
            continue

        current_id = normalize_id(
            source.get("id")
        )

        # id 不能为空
        if not current_id:
            continue

        # 排除 file3 中已有的 id
        if current_id in file3_ids:
            continue

        item = {
            "id": current_id,
            "title": clean_text(
                source.get("title")
            ),
            "content": clean_text(
                source.get("content")
            ),
            "label": "无用"
        }

        useless_candidates.append(item)

    print(
        f"is_show 为空且 id 不在 file3 中："
        f"{len(useless_candidates)} 条"
    )

    # ==========================================
    # 5. 检查数量
    # ==========================================

    if len(useless_candidates) < useful_count:

        raise ValueError(
            "\n无用候选数据不足！\n"
            f"有用数据：{useful_count} 条\n"
            f"无用候选：{len(useless_candidates)} 条\n"
            f"还缺："
            f"{useful_count - len(useless_candidates)} 条"
        )

    # ==========================================
    # 6. 随机抽取同样数量
    # ==========================================

    random.seed(random_seed)

    useless_data = random.sample(
        useless_candidates,
        useful_count
    )

    print(
        f"随机抽取无用数据：{len(useless_data)} 条"
    )

    # ==========================================
    # 7. 合并
    # ==========================================

    result = useful_data + useless_data

    # ==========================================
    # 8. 打乱
    # ==========================================

    random.seed(random_seed)
    random.shuffle(result)

    # ==========================================
    # 9. 保存 JSONL
    # ==========================================

    print(
        f"\n正在保存：{output_path}"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for item in result:

            line = json.dumps(
                item,
                ensure_ascii=False
            )

            f.write(line + "\n")

    # ==========================================
    # 10. 输出统计
    # ==========================================

    print("\n==============================")
    print("处理完成")
    print("==============================")
    print(f"有用数据：{len(useful_data)} 条")
    print(f"无用数据：{len(useless_data)} 条")
    print(f"总数据量：{len(result)} 条")
    print(f"输出文件：{output_path}")


if __name__ == "__main__":

    build_dataset(
        file2_path="/mnt/cognitive_classification/data_process/DQDX_no_hand.json",
        file3_path="/mnt/cognitive_classification/data/train.jsonl",
        output_path="/mnt/cognitive_classification/data/test.jsonl",
        random_seed=42
    )

正在读取 file3.jsonl ...
file3.jsonl 中读取到 424 个不同 id

正在读取 file2.json ...
file2.json 总数据量：1000 条
is_show = 2 的有用数据：13 条
is_show 为空且 id 不在 file3 中：729 条
随机抽取无用数据：13 条

正在保存：/mnt/cognitive_classification/data/test.jsonl

处理完成
有用数据：13 条
无用数据：13 条
总数据量：26 条
输出文件：/mnt/cognitive_classification/data/test.jsonl
